# ランダムなエルミート行列の量子位相推定

著者: Takumi Kato (Blueqat inc.), Maho Nakata (Riken), Shinya Morino, Seiya Sugo (Quemix inc.), Yuichiro Minato (Blueqat inc.)

[前回](301_pea.ipynb)は、Z ゲートと X ゲートの行列の固有値を計算しました。今回は、ランダムに生成した 2x2 のエルミート行列に対する量子位相推定を行います。

エルミート行列の固有値を計算することは、量子力学における物理量を求める上で有用であり、量子化学や量子シミュレーションといった分野で幅広く応用されています。

## 固有値の和がゼロになるエルミート行列の量子位相推定

エルミート行列に対する量子位相推定の基本的な考え方は、前回と同じです。前回は、Z ゲートと X ゲートを推定するために Controlled-U ゲートとして Controlled-Z ゲートと Controlled-X ゲートを準備しました。しかし、一般にエルミート行列はユニタリ行列ではありません。したがって、Controlled-U ゲートを直接準備することはできません。

結論から言うと、エルミート行列 $\hat H$ に対する Controlled-U ゲートとして Controlled-$e^{2\pi i\hat H}$ ゲートと、$\hat H$ の固有ベクトルを与える量子回路を準備する必要があります。この回路とゲートを用いることで、量子位相推定を行い、固有値を得ることができます。

以下の説明は少し難しいので、読み飛ばしてコードを読んでも構いません。理解するためには[行列指数関数](https://en.wikipedia.org/wiki/Matrix_exponential)を知っておく必要があります。

$\hat H$ を、固有値の和が 0 になるエルミート行列とします。

$\hat H$ が固有値と固有ベクトルの組 $\{(\lambda_j, \left|\psi_j\right\rangle)\}$ を持つとき、すべての $j$ について $$\hat H\left|\psi_j\right\rangle = \lambda_j \left|\psi_j\right\rangle$$ が成り立ちます。
$\hat H$ はエルミートであるため $\{\lambda_j\}$ は実数です。

前回は $\hat H = Z$ または $\hat H = X$ であり、位相キックバックのために Controlled-Z ゲートまたは Controlled-X ゲートを使いました。しかし一般には、$\hat H$ はユニタリ行列ではないため Controlled-$\hat H$ ゲートを作ることができず、別の方法を見つける必要があります。

ここでは、ユニタリ行列 $U = e^{2\pi i\hat H}$ の量子位相推定を考えます。

### $U = e^{2\pi i\hat H}$ はユニタリ行列か?

$U U^\dagger = I$ を証明するために、$U^\dagger$ を考えます。
$$\begin{eqnarray}
U^\dagger &=& (e^{2\pi i\hat H})^\dagger\nonumber\\
&=& (\sum_n^\infty \frac{(2\pi i\hat H)^n}{n!})^\dagger\nonumber\\
&=& \sum_n^\infty \frac{((2\pi i\hat H)^n)^\dagger}{n!}\nonumber\\
&=& \sum_n^\infty \frac{(-2\pi i\hat H^\dagger)^n)}{n!}\nonumber\\
&=& \sum_n^\infty \frac{(-2\pi i\hat H)^n)}{n!}\nonumber\\
&=& e^{-2\pi i\hat H}.\nonumber
\end{eqnarray}$$
したがって、$U U^\dagger = e^{2\pi i\hat H} e^{-2\pi i\hat H}$ です。$[2\pi i\hat H, -2\pi i\hat H] = 0$ であるため、$e^{2\pi i\hat H} e^{-2\pi i\hat H} = e^{2\pi i\hat H -2\pi i\hat H} = e^{0\hat H} = I$ となります。よって $U U^\dagger = I$、すなわち $U$ はユニタリ行列です。

### $U$ の固有値と固有ベクトルは何か? $\hat H$ のものと関係があるか?

$U$ の固有ベクトルが $\hat H$ の固有ベクトルと同じであることを証明し、対応する固有値を求めます。

$\lambda_j, \left|\psi_j\right\rangle$ を $\hat H$ の固有値と固有ベクトルの組とします。すると、
$$\begin{eqnarray}
U \left|\psi_j\right\rangle &=& (\sum_n^\infty \frac{(2\pi i\hat H)^n}{n!})\left|\psi_j\right\rangle\nonumber\\
&=& \sum_n^\infty \frac{(2\pi i)^n \hat H^n \left|\psi_j\right\rangle}{n!}\nonumber\\
&=& \sum_n^\infty \frac{(2\pi i)^n \lambda_j^n \left|\psi_j\right\rangle}{n!}\nonumber\\
&=& \sum_n^\infty \frac{(2\pi i)^n \lambda_j^n}{n!}\left|\psi_j\right\rangle\nonumber\\
&=& \sum_n^\infty \frac{(2\pi i \lambda_j)^n}{n!}\left|\psi_j\right\rangle\nonumber\\
&=& e^{2\pi i \lambda_j}\left|\psi_j\right\rangle.\nonumber
\end{eqnarray}$$
したがって、$\left|\psi_j\right\rangle$ は $U$ の固有ベクトルであり、対応する固有値は $e^{2\pi i \lambda_j}$ です。

これにより、
$$U\left|\psi_j\right\rangle = e^{2\pi i\lambda_j} \left|\psi_j\right\rangle$$
の量子位相推定を行うことで、$\hat H$ の固有値 $\lambda_j$ を得ることができます。

## 実装
まず、必要なライブラリをインポートします。今回はデフォルトの(tensornet)バックエンドを使用します。

In [ ]:
import math
import cmath
import random
import numpy as np

from blueqat import *
from blueqat.utils import X, Y, Z, I
from blueqat.circuit_funcs import circuit_to_unitary

pi = math.pi

# Version check for Blueqat
try:
    Circuit().r(0.1)[0].run()
except AttributeError:
    raise ImportError('Blueqat version is old.')

次に、ランダムなエルミート行列 $\hat H$ を作成します。

量子位相推定によって固有値を得るためには、以下も必要です。
- 求めたい固有値に対応する $\hat H$ の固有ベクトルを与える量子回路
- Controlled-$e^{i2\pi \hat H 2^n}$ を与える量子回路

これらを作成していきます。

エルミート行列は $\hat H = P D P^\dagger$ のように分解できます。ここで $P$ はユニタリ行列、$D$ は実対角行列です。

この場合、$D$ の対角成分は $\hat H$ の固有値であり、$P$ の各列は $\hat H$ の固有ベクトルです。ここでは、固有値の和がゼロになる 2x2 のエルミート行列を考えているので、$\hat H$ が固有値 $E$ を持つとき、$D$ の要素は $\pm E$ となります。

そこで、ランダムな実数 $E$ を生成して $D$ を作り、次にランダムな 3 つの実数 $\theta, \phi, \lambda$ を生成して、U ゲートを用いて $P$ を作ります。すなわち $P = \mathrm{U}(\theta, \phi, \lambda)$ です。ここで、U ゲートのパラメータ $\gamma$ は固有値に無関係なので無視します。

それでは、以下を返す関数を定義しましょう。

- エルミート行列 $\hat H$
- 求めたい固有値 $E$
- 固有ベクトルを与える U ゲートのパラメータ $\theta, \phi, \lambda$

In [ ]:
def is_hermitian(mat):
    """Check whether mat is Hermitian"""
    return np.allclose(mat, mat.T.conjugate())

def get_u_matrix(theta, phi, lam):
    """Get a unitary matrix U(theta, phi, lam)"""
    # Get the unitary matrix via circuit_to_unitary() and convert to numpy matrix
    u = circuit_to_unitary(Circuit().u(theta, phi, lam)[0])
    return np.array(u).astype(np.complex64)

def random_hermitian():
    """Make random Hermitian and returns triplet
    (Hermitian, eigenvalue, parameters for U).
    """
    # Generate random eigenvalue
    eigval = random.random()
    # Generate random parameters for U
    theta = random.random()
    phi = random.random()
    lam = random.random()
    # Make Hermitian from them
    u = get_u_matrix(theta, phi, lam)
    hermitian = u @ np.diag([eigval, -eigval]) @ u.T.conjugate()
    # Check it is Hermitian
    assert is_hermitian(hermitian)
    # returns Hermitian, eigenvalue, parameters for U
    return hermitian, eigval, (theta, phi, lam)

では、エルミート行列を作ってみましょう。

In [ ]:
H, E, (theta, phi, lam) = random_hermitian()
print(H)

`theta, phi, lam` と U ゲートを使って固有ベクトルを作ることができます。

In [ ]:
vec = Circuit().u(theta, phi, lam)[0].run()
print(vec)

`H vec = E vec` を確認します。これは、`E, vec` が H の固有値と固有ベクトルの組であることを意味します。

In [ ]:
np.allclose(np.dot(H, vec), E * vec)

準備が整いました。量子位相推定を実装していきます。量子回路を作り、量子位相推定によって `E` を計算します。

In [ ]:
def iqft(c, q0, n_qubits):
    """Add inversed quantum Fourier transform operations to q0-th - (q0 + n_qubits)-th qubits of the circuit `c`"""
    for i in reversed(range(n_qubits)):
        angle = -0.5
        for j in range(i + 1, n_qubits):
            c.cr(angle * pi)[q0 + j, q0 + i]
            angle *= 0.5
        c.h[q0 + i]
    return c


def initial_circuit(theta, phi, lam):
    """Prepare a initial circuit which gives eigenvector."""
    return Circuit().u(theta, phi, lam)[0]


def apply_cu(c, ctrl, theta, phi, lam, eigval, n):
    """Append Controlled-U^(2^n) to the circuit `c`.
    Controll qubit is `ctrl`, target qubit is 0.
    
    This function requires eigenvalue `eigval` as an argument.
    We make Controlled-U^(2^n) by using eigenvalue. You may feel we're cheating.
    You can make approximate Controlled-U^(2^n) circuit without eigenvalue,
    for example by using Suzuki-Trotter decomposition. In this case, you have to consider about precision.
    Generally, making efficient and high-precision Controlled-U^(2^n) circuit without cheating is difficult.
    """
    return c.u(-theta, -lam, -phi)[0].crz(-2 * pi * eigval * (2**n))[ctrl, 0].u(theta, phi, lam)[0]

def qpe_circuit(eigval, theta, phi, lam, precision):
    """Returns quantum phase estimation circuit from eigenvalue, parameters of U gate and
    precision (a number of binary digits) of phase estimation"""
    c = initial_circuit(theta, phi, lam)
    c.h[1:1 + precision]
    for i in range(precision):
        apply_cu(c, i + 1, theta, phi, lam, eigval, i)
    iqft(c, 1, precision)
    return c

量子位相推定の量子回路を見てみましょう。

In [ ]:
%matplotlib inline
qpe_circuit(E, theta, phi, lam, 4).run_with_draw()

次に、観測結果から固有値を計算する関数を作ります。

In [ ]:
def run_qpe(c, shots=1000, max_candidates=5):
    """Run the circuit for quantum phase estimation and returns candidates of eigenvalue.
    shots: The number of running quantum circuit, max_candidates: Maximum number of candidates
    """
    # Use the explicit "statevector" backend: the default "tensornet" backend can run out of
    # memory / crash for the larger circuits built at high precision below.
    cnt = c.m[1:].run(shots=shots, backend="statevector")

    # Convert measured result to floating point value
    def to_value(k):
        # The new SDK's shot-result bitstrings are ordered with qubit 0 as the rightmost
        # character (opposite of the old convention), so reverse before parsing.
        k = k[::-1]
        k = k[1:] # Drop first element
        val = 0 # Value
        a = 1.0 
        for ch in k:
            if ch == '1':
                val += a
            a *= 0.5
        if val > 1:
            # When the phase > π, subtract 2π (phase is negative).
            val = val - 2
        return val

    return [(to_value(k), v) for k, v in cnt.most_common(max_candidates)]

これで量子位相推定を実行できます。結果を見てみましょう。

In [ ]:
print('Eigenvalue (expected):', E) # It's expected value. We desire a value closed to it.
# We'll run small to high precision for comparison.
for precision in range(3, 16):
    print(precision, 'bit precision:')
    c = qpe_circuit(E, theta, phi, lam, precision)
    result = run_qpe(c, 1000, 3)
    for value, count in result:
        # Display a number of observation, obtained eigenvalue and a deviation from true eigenvalue.
        print(f'{count:<5}{value:<18}(deviation: {value - E: .3e})')
    print('')

得られた値は真の固有値に近い値になっています。

## エルミート行列の量子位相推定(固有値の和がゼロにならない場合)
一般に、エルミート行列の固有値の和はゼロにはなりません。この場合も量子位相推定によって固有値を計算できますが、Controlled-U ゲートの回路を修正する必要があります。

$U$ を $e^{2\pi i\hat H}$ とします。$U$ の固有値の和は $U$ のトレースと等しくなります。そこで $\mathrm{tr}(U)$ を考えます。
$$\begin{eqnarray}
\mathrm{tr}(U) &=& \mathrm{tr}\left(\sum_{n=0}^{\infty}\frac{(2\pi i \hat H)^n}{n!}\right)\nonumber\\
&=&\sum_{n=0}^{\infty}\mathrm{tr}\left(\frac{(2\pi i \hat H)^n}{n!}\right)\nonumber\\
&=&\sum_{n=0}^{\infty}\frac{(2\pi i \mathrm{tr}(\hat H))^n}{n!}\nonumber\\
&=&e^{2\pi i \mathrm{tr}(\hat H)}\nonumber.\\
\end{eqnarray}$$
したがって、$\mathrm{tr}(U)$ は $U$ の大域位相として現れます。この大域位相を考慮して Controlled-U ゲートを作ります。すなわち、制御量子ビットに RZ ゲートを追加します。

In [ ]:
def random_hermitian2():
    """Make random Hermitian matrix and returns
    Hermitian matrix, a pair of eigenvalues, parameters for U gate.
    """
    # Get random eigenvalues. The range of eigenvalues is -1 to 1.
    eigvals = [random.random() * 2 - 1, random.random() * 2 - 1]
    # Sorting eigenvalues. (You may comment out this line if you don't desire.)
    eigvals.sort()
    # Get random parameters for U gate.
    theta = random.random()
    phi = random.random()
    lam = random.random()
    # Make a Hermitian matrix from them.
    u = get_u_matrix(theta, phi, lam)
    hermitian = u @ np.diag(eigvals) @ u.T.conjugate()
    # Check if it is Hermitian.
    assert is_hermitian(hermitian)
    # Returns Hermitian, eigenvalues, parameters.
    return hermitian, eigvals, (theta, phi, lam)

In [ ]:
H, eigvals, (theta, phi, lam) = random_hermitian2()
print(H)

In [ ]:
def apply_cu2(c, ctrl, theta, phi, lam, eigvals, n):
    """Modify apply_cu and consider global phase also.
    argument `eigvals` is a pair of eigenvalues.
    """
    bias = sum(eigvals) / 2
    angle = (eigvals[0] - eigvals[1]) / 2
    return c.u(-theta, -lam, -phi)[0].crz(-2 * pi * angle * (2**n))[ctrl, 0].u(theta, phi, lam)[0].rz(pi * bias * (2**n))[ctrl]

def qpe_circuit2(eigvals, theta, phi, lam, precision):
    """Modify qpe_circuit2 for using apply_cu2."""
    c = initial_circuit(theta, phi, lam)
    c.h[1:1 + precision]
    for i in range(precision):
        apply_cu2(c, i + 1, theta, phi, lam, eigvals, i)
    iqft(c, 1, precision)
    return c

次に、最初の固有値を計算します。

In [ ]:
print('Eigenvalue (expected):', eigvals[0]) # It's expected value. We desire a value closed to it.
# We'll run small to high precision for comparison.
for precision in range(3, 16):
    print(precision, 'bit precision:')
    c = qpe_circuit2(eigvals, theta, phi, lam, precision)
    result = run_qpe(c, 1000, 3)
    for value, count in result:
        # Display a number of observation, obtained eigenvalue and a deviation from true eigenvalue.
        print(f'{count:<5}{value:<18}(deviation: {value - eigvals[0]: .3e})')
    print('')

Controlled-U ゲートを修正することで、固有値を得ることができました。

次に、別の固有値を求めることを考えます。別の固有値は、別の固有ベクトルを用いた量子位相推定によって計算できます。

今回は $\hat H = P D P^\dagger$ という式を使います。ここで $P$ はユニタリ行列 $\mathrm{U}(\theta, \phi, \lambda)$ です。したがって、$\hat H$ の固有ベクトルは $\mathrm{U}(\theta, \phi, \lambda)$ の1列目と2列目です。最初の固有値を求めるために、$\mathrm{U}(\theta, \phi, \lambda)$ ゲートを適用して最初の固有ベクトルを準備しました。2番目の固有ベクトルは、$\mathrm{U}(\theta, \phi, \lambda)$ ゲートの前に X ゲートを適用することで得ることができます。

実装してみましょう。

In [ ]:
def initial_circuit2(theta, phi, lam):
    """Make an initial circuit which prepares second eigenvector."""
    return Circuit().x[0].u(theta, phi, lam)[0]

def qpe_circuit3(eigvals, theta, phi, lam, precision):
    """Modified for using apply_cu2 and initial_circuit2."""
    c = initial_circuit2(theta, phi, lam)
    c.h[1:1 + precision]
    for i in range(precision):
        apply_cu2(c, i + 1, theta, phi, lam, eigvals, i)
    iqft(c, 1, precision)
    return c

In [ ]:
print('Eigenvalue (expected):', eigvals[1]) # It's expected value. We desire a value closed to it.
# We'll run small to high precision for comparison.
for precision in range(3, 16):
    print(precision, 'bit precision:')
    c = qpe_circuit3(eigvals, theta, phi, lam, precision)
    result = run_qpe(c, 1000, 3)
    for value, count in result:
        # Display a number of observation, obtained eigenvalue and a deviation from true eigenvalue.
        print(f'{count:<5}{value:<18}(deviation: {value - eigvals[1]: .3e})')
    print('')

2番目の固有値も得ることができました。